# Data Exploration
This notebook explores all the clinical and genomic files provided in the `data/` directory. It gives us a sense of what we are working with before we begin building the bioinformatics pipeline.

In [1]:
import os
from pathlib import Path
import pandas as pd

# Let's see what is in the data directory and how large the files are
data_dir = Path('../data')
files = list(data_dir.glob('*.*'))

file_info = []
for f in sorted(files):
    size_gb = f.stat().st_size / (1024**3)
    size_mb = f.stat().st_size / (1024**2)
    
    size_str = f"{size_gb:.2f} GB" if size_gb > 1 else f"{size_mb:.2f} MB"
    file_info.append({'File': f.name, 'Size': size_str})

display(pd.DataFrame(file_info))

,File,Size
0,.cache,0.00 MB
1,Challenge_Clinical_Phenotype_1.docx,0.02 MB
2,WGS_EX2312012_HGWCNDSX7.vcf.gz,300.55 MB
3,WGS_EX2312012_HGWCNDSX7.vcf.gz.tbi,2.23 MB
4,WGS_EX2312012_HGWCNDSX7_S16_L001_R1_001.fastq.gz,9.75 GB
5,WGS_EX2312012_HGWCNDSX7_S16_L001_R2_001.fastq.gz,10.28 GB
6,WGS_EX2312012_HGWCNDSX7_S16_L002_R1_001.fastq.gz,9.66 GB
7,WGS_EX2312012_HGWCNDSX7_S16_L002_R2_001.fastq.gz,10.16 GB
8,WGS_EX2312012_HGWCNDSX7_S16_L003_R1_001.fastq.gz,9.50 GB
9,WGS_EX2312012_HGWCNDSX7_S16_L003_R2_001.fastq.gz,9.89 GB


## Processed Genomic Data (.vcf.gz)
The VCF file contains the **genotype** differences. We will use `cyvcf2` to inspect the header, verify the sample name, and peek at the first 3 variant calls.

### Understanding the Columns & Coordinates
When we load the VCF into a pandas DataFrame, we extract the most critical columns for each variant:

*   **`CHROM` (Chromosome):** Which of the 23 chromosome pairs (or X/Y sex chromosomes) the variant is located on.
*   **`POS` (Position):** The exact coordinate (base-pair number) on that chromosome.
*   **`ID`:** The variant identifier, often a dbSNP RSID (e.g., `rs12345`). Usually `.` if unannotated.
*   **`REF` (Reference Allele):** The DNA letter(s) that a standard human reference genome has at this exact position.
*   **`ALT` (Alternate Allele):** The DNA letter(s) that **this specific child** actually has.
*   **`QUAL` (Quality Score):** A numerical score showing how confident the sequencing machine and algorithm are that this mutation is real and not just a scanning error. A higher number is better.
*   **`FILTER`:** Did this variant pass the algorithm's quality control checks? Variants with specific failure flags (e.g., `LowQual`) should be filtered out. Good variants often have `.` or `PASS`.
*   **`INFO_DP` (Total Depth):** Combined depth of reads covering this position.
*   **`INFO_MQ` (Mapping Quality):** RMS mapping quality. Higher indicates reads are unambiguously mapped to this genomic region.
*   **`INFO_FS` (Fisher Strand):** Phred-scaled p-value to detect strand bias. High values suggest sequencing artifacts.
*   **`FORMAT_GT` (Genotype):** Tells us *how* the child inherited this mutation. `1` = Heterozygous (one normal copy, one mutated copy). `3` = Homozygous Alternate (two mutated copies).
*   **`FORMAT_DP` (Sample Depth):** The number of sequencing reads covering this position specifically for this sample.
*   **`FORMAT_GQ` (Genotype Quality):** Confidence score specifically for the Genotype (`FORMAT_GT`) call.
*   **`FORMAT_AD_REF` & `FORMAT_AD_ALT` (Allele Depths):** The exact number of sequencing reads supporting the reference allele vs the alternate allele. Critical for verifying if a heterozygous call is truly ~50/50.


In [6]:
from cyvcf2 import VCF
import pandas as pd

vcf_path = '../data/WGS_EX2312012_HGWCNDSX7.vcf.gz'
vcf = VCF(vcf_path)
print(f'Samples in VCF: {vcf.samples}')

# Peek at the first 3 variants
records = []
for i, variant in enumerate(vcf):
    records.append({
        'CHROM': variant.CHROM,
        'POS': variant.POS,
        'ID': variant.ID,
        'REF': variant.REF,
        'ALT': ",".join(variant.ALT) if variant.ALT else "",
        'QUAL': variant.QUAL,
        'FILTER': variant.FILTER,
        'INFO_DP': variant.INFO.get('DP'),
        'INFO_MQ': variant.INFO.get('MQ'),
        'INFO_FS': variant.INFO.get('FS'),
        'FORMAT_GT': variant.gt_types[0], # 0=HOM_REF, 1=HET, 2=UNKNOWN, 3=HOM_ALT
        'FORMAT_DP': variant.format('DP')[0][0] if variant.format('DP') is not None else None,
        'FORMAT_GQ': variant.format('GQ')[0][0] if variant.format('GQ') is not None else None,
        'FORMAT_AD_REF': variant.format('AD')[0][0] if variant.format('AD') is not None else None,
        'FORMAT_AD_ALT': variant.format('AD')[0][1] if variant.format('AD') is not None else None
    })

display(pd.DataFrame(records))

Samples in VCF: ['WGS_EX2312012']


,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO_DP,INFO_MQ,INFO_FS,FORMAT_GT,FORMAT_DP,FORMAT_GQ,FORMAT_AD_REF,FORMAT_AD_ALT
0,1,13273,NaN,G,C,385.769989,MQ40,13,33.290001,0.0,3,13,39,0,13
1,1,13417,NaN,C,CGAGA,320.750000,MQ40,9,31.660000,0.0,3,8,24,0,8
2,1,14248,NaN,T,G,59.770000,MQ40,13,30.100000,0.0,1,13,88,8,5
3,1,14354,NaN,C,A,11.120000,LowQual;MQ40;QD2,16,32.270000,0.0,1,16,39,13,3
4,1,14590,NaN,G,A,76.769997,MQ40,10,24.129999,0.0,1,10,99,7,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5012199,Un_JTFH01001998v1_decoy,1421,NaN,G,A,1487.770020,NaN,35,59.290001,0.0,3,34,99,0,34
5012200,Un_JTFH01001998v1_decoy,1546,NaN,T,C,846.770020,NaN,26,59.849998,0.0,3,25,75,0,25
5012201,Un_JTFH01001998v1_decoy,1613,NaN,A,T,376.769989,NaN,33,59.880001,0.0,1,31,99,17,14
5012202,Un_JTFH01001998v1_decoy,1626,NaN,T,A,1101.770020,NaN,34,59.889999,0.0,3,30,80,1,29


In [7]:
df = pd.DataFrame(records)
del records

In [9]:
df.FORMAT_GT.value_counts()

FORMAT_GT
1    3113937
3    1898267
Name: count, dtype: int64

## Raw Genomic Data (.fastq.gz)
These are the massive raw sequencing files (~10GB each). We won't load them into memory, but we can use `gzip` to peek at the first 4 lines of one file.

In a FASTQ file, exactly **4 lines represent 1 single sequencing read**:
1. A sequence identifier with information about the sequencing run.
2. The sequence itself (the A, C, T, G, and N characters).
3. A separator line (starts with `+`).
4. The base quality scores (ASCII characters representing how confident the machine is in each letter).

In [ ]:
import gzip
import os

fastq_path = '../data/WGS_EX2312012_HGWCNDSX7_S16_L001_R1_001.fastq.gz'
print(f'Reading the first 4 lines (1 read) of {os.path.basename(fastq_path)}:\n')
with gzip.open(fastq_path, 'rt') as f:
    for i in range(4):
        print(f.readline().strip())